### Attribute and instance

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        # Call the constructor of the parent class nn.Module to perform
        # the necessary initialization
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    # Define the forward propagation of the model, that is, how to return the
    # required model output based on the input X
    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

What self actually means

In Python OOP:

* self = the current object (instance) of the class
* self.x = an attribute belonging to that specific object

So when you write:

```
self.hidden = nn.LazyLinear(256)
self.out = nn.LazyLinear(10)
```

you are:
👉 Creating attributes of this MLP instance
👉 Storing layer objects inside the model

Are self.hidden and self.out instances?

Yes — in this case:
```
self.hidden = nn.LazyLinear(256)
```

`nn.LazyLinear(256)` is an instance of a class (`LazyLinear`)
So `self.hidden` is an instance stored as an attribute

Same for `self.out`.

### How `nn.Sequential` actually works under the hood

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self.add_module(str(idx), module)

    def forward(self, X):
        for module in self.children():
            X = module(X)
        return X

Why introduce `MySequential` at all?

The goal is not to replace `nn.Sequential` in practice.

It’s to teach you how PyTorch actually works under the hood.

What `nn.Sequential` does (conceptually)

```
nn.Sequential(layer1, layer2, layer3)
```

is just a container that:

1. Stores layers in order
2. Registers them as submodules
3. Applies them one by one in forward pass

What `MySequential` teaches you

This implementation reveals two key PyTorch mechanisms:

(A) How modules are registered
```
self.add_module(name, module)
```

👉 This tells PyTorch:

* “This layer belongs to the model”
* “Track its parameters”
* “Include it in .parameters() and .state_dict()”

Without this step, the model is invisible to PyTorch’s training system.

(B) How forward propagation is just iteration
```
for module in self.children():
    X = module(X)
```

👉 This shows:

* A model is just a chain of function calls
* `children()` returns all registered submodules in order

Why do we need `idx` from `enumerate(args)`?

This is the subtle but important part.

Look at this line:
```
self.add_module(str(idx), module)
```

Why do we need a name?

Because:

👉 PyTorch stores submodules in a dictionary-like structure

Internally, it looks like:
```
self._modules = {
    "0": module1,
    "1": module2,
    ...
}
```
So every module must have a unique string name

Why use `idx`?

Because:

* `args` is just a tuple: (layer1, layer2, layer3)
* It has no names
* We must assign names ourselves

So:
```
for idx, module in enumerate(args):
```
gives:
```
(0, layer1)
(1, layer2)
(2, layer3)
```
Then:
```
self.add_module("0", layer1)
self.add_module("1", layer2)
```